# 06 — Just in time

The lens graph in chapter 04 was wired by hand. This chapter deletes the wiring and lets the
incident decide.

> **You'll learn**
> - Have a reasoner **choose** its own fan-out from a 26-lens vocabulary at runtime
> - Cap that fan-out in code, because a model asked "which apply?" will answer "all of them"
> - See two incidents grow two **visibly different** graphs from the same source file

Same cockpit as every chapter: drive the node over HTTP through the control plane, never
`await app.call(...)` from a cell.

In [1]:
import sys, json, requests
sys.path.insert(0, "../lib")
import dag

SERVER = "http://localhost:8080"

def run(reasoner, **inp):
    r = requests.post(f"{SERVER}/api/v1/execute/blast-radius.{reasoner}",
                      json={"input": inp}, timeout=600)
    r.raise_for_status()
    return r.json()

print(dag.RUN_ENDPOINT)

/api/v1/agentic/run/{run_id}


The vocabulary is `incidents/lenses.md`, transcribed into `node/rungs/r06.py`. Twenty-six
standing questions. Nobody says in advance which ones this incident needs.

In [2]:
sys.path.insert(0, "../node")
from rungs.r06 import LENSES, MAX_LENSES

print(f"{len(LENSES)} lenses in the vocabulary, fan-out capped at {MAX_LENSES}")
print()
for lid, q in list(LENSES.items())[:6]:
    print(f"  {lid:22} {q}")
print("  ...")

26 lenses in the vocabulary, fan-out capped at 6

  alert_validity         Is the alert real, or did the monitor change?
  blast_scope            Who is affected, and more importantly who is not?
  timeline               When did it truly start, and does the candidate cause precede it?
  change_correlation     What changed inside the window?
  long_horizon_change    What changed days or weeks ago that is only now visible?
  rollback_viability     Can we undo this safely, or has state moved?
  ...


The cap is enforced in code, not asked for in the prompt. An uncapped generator rebuilds the
hand-wired graph with extra steps.

In [3]:
import inspect
from rungs import r06
src = inspect.getsource(r06.choose_lenses)
print(src[src.index("    # Enforce"):])

    # Enforce the cap and the vocabulary in code. A model told "at most six" will
    # sometimes return nine, and will sometimes invent a lens that sounds right.
    picked = [l for l in dict.fromkeys(plan.lenses) if l in LENSES][:MAX_LENSES]
    plan.lenses = picked or ["timeline", "change_correlation", "blast_scope"]
    return plan



**inc-011** is a clock-skew incident: 401s on one pod, no deploy, a live-migration hours
earlier. Watch which lenses it opens.

In [4]:
a = run("r06_diagnose", incident_id="inc-011")
plan_a = run("r06_plan_only", incident_id="inc-011")["result"]

print("inc-011 chose:", plan_a["lenses"])
print("       shut  :", plan_a["ruled_out"])
print()
print(plan_a["reasoning"])

inc-011 chose: ['clock_time', 'blast_scope', 'timeline', 'change_correlation']
       shut  : ['alert_validity', 'dependency_health', 'security_abuse', 'credential_expiry', 'cache_behavior']

Clock_time: metrics show node eu-c1-n07 clock offset 92.5s and chronyd unsynchronized, causing 'token used before issued' JWT rejections on pod booking-api-77bd-c3. Blast_scope: only pod c3 on node eu-c1-n07 sees failures; other pods succeed and per-pod error rate metric shows c3 at 96% while a1 is 0.6%, pinpointing a single node. Timeline: alert fired at 14:22 but errors began ~14:08, matching the pod c3 error rate spike, and the clock drift likely started after the hypervisor live-migration window that completed at 02:40Z on the same node. Change_correlation: the hypervisor live-migration on eu-c1-n07 is the only infrastructure change that could have disrupted NTP sync and caused the clock skew on that specific node.


**inc-006** is a cache stampede. Different symptom shape, so a different set of standing
questions is worth asking.

In [5]:
b = run("r06_diagnose", incident_id="inc-006")
plan_b = run("r06_plan_only", incident_id="inc-006")["result"]

print("inc-006 chose:", plan_b["lenses"])
print("       shut  :", plan_b["ruled_out"])
print()
print(plan_b["reasoning"])

inc-006 chose: ['cache_behavior', 'change_correlation', 'database_health', 'connection_pool', 'retry_amplification']
       shut  : ['alert_validity', 'external_vendor', 'security_abuse', 'config_drift', 'long_horizon_change']

cache_behavior: The cache hit ratio collapsed from 97% to under 4% because the deploy changed the key namespace from pcat:v7 to pcat:v8, invalidating the entire cache. change_correlation: The deploy at 04:10:00Z changed the cache key namespace, which is the candidate cause that aligns exactly with the start of the database load spike. database_health: Although the database is healthy nominally, the sudden surge to 11,400 queries per second from cache misses saturated its CPU to 100% and triggered known seq scans on product_variant. connection_pool: The 500-connection pool was exhausted within minutes as all replicas tried to fetch from the database simultaneously, producing connection pool timeout errors. retry_amplification: Edge-proxy retries (rising from 2 to

Neither incident opened the other's lens. That is the claim of this chapter, in one cell.

In [6]:
A, B = set(plan_a["lenses"]), set(plan_b["lenses"])
print(f"shared      : {sorted(A & B)}")
print(f"inc-011 only: {sorted(A - B)}")
print(f"inc-006 only: {sorted(B - A)}")
print()
print("clock_time    in inc-011:", "clock_time" in A, "| in inc-006:", "clock_time" in B)
print("cache_behavior in inc-011:", "cache_behavior" in A, "| in inc-006:", "cache_behavior" in B)

shared      : ['change_correlation']
inc-011 only: ['blast_scope', 'clock_time', 'timeline']
inc-006 only: ['cache_behavior', 'connection_pool', 'database_health', 'retry_amplification']

clock_time    in inc-011: True | in inc-006: False
cache_behavior in inc-011: False | in inc-006: True


And here are the two graphs the control plane actually recorded. Same code, same file, one
run apart.

In [7]:
dag.render_two(a["run_id"], b["run_id"],
               labels=("inc-011 · clock skew", "inc-006 · cache stampede"))

```mermaid
flowchart LR
  subgraph ag["inc-011 · clock skew — 7 exec · depth 2 · fan-out 6"]
  direction TD
    a0["r06_diagnose<br/><small>✓ succeeded · 33.8s</small>"]
    a1["r06_choose_lenses<br/><small>✓ succeeded · 12.9s</small>"]
    a2["r06_apply_lens<br/><small>✓ succeeded · 5.1s</small>"]
    a3["r06_apply_lens<br/><small>✓ succeeded · 10.8s</small>"]
    a4["r06_apply_lens<br/><small>✓ succeeded · 4.5s</small>"]
    a5["r06_apply_lens<br/><small>✓ succeeded · 4.9s</small>"]
    a6["r06_apply_lens<br/><small>✓ succeeded · 5.1s</small>"]
    a0 --> a1
    a0 --> a2
    a0 --> a3
    a0 --> a4
    a0 --> a5
    a0 --> a6
    class a0,a1,a2,a3,a4,a5,a6 ok;
  end
  subgraph bg["inc-006 · cache stampede — 8 exec · depth 2 · fan-out 7"]
  direction TD
    b0["r06_diagnose<br/><small>✓ succeeded · 39.1s</small>"]
    b1["r06_choose_lenses<br/><small>✓ succeeded · 17.6s</small>"]
    b2["r06_apply_lens<br/><small>✓ succeeded · 4.5s</small>"]
    b3["r06_apply_lens<br/><small>✓ succeeded · 6.8s</small>"]
    b4["r06_apply_lens<br/><small>✓ succeeded · 4.5s</small>"]
    b5["r06_apply_lens<br/><small>✓ succeeded · 4.5s</small>"]
    b6["r06_apply_lens<br/><small>✓ succeeded · 7.9s</small>"]
    b7["r06_apply_lens<br/><small>✓ succeeded · 6.8s</small>"]
    b0 --> b1
    b0 --> b2
    b0 --> b3
    b0 --> b4
    b0 --> b5
    b0 --> b6
    b0 --> b7
    class b0,b1,b2,b3,b4,b5,b6,b7 ok;
  end
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

The diagnoses, for the record. Both name the mechanism, not the service.

In [8]:
for label, res in (("inc-011", a["result"]), ("inc-006", b["result"])):
    print(f"--- {label} ---")
    print("root cause :", res["root_cause"])
    print("remediation:", res["remediation"])
    print("confident  :", res["confident"], f"({len(res['findings'])} findings)")
    print()

--- inc-011 ---
root cause : Live-migration of hypervisor on node eu-c1-n07 caused a 92-second clock skew, making booking-api-77bd-c3 reject valid JWTs and webhook signatures due to 'token used before issued' errors.
remediation: Synchronize the clock on node eu-c1-n07 by restarting chronyd or forcing an NTP sync (e.g., 'chronyc makestep'), then verify all pods on that node are healthy. Alternatively, reschedule pod booking-api-77bd-c3 to a different node via pod eviction.
confident  : True (7 findings)

--- inc-006 ---
root cause : The deploy dep-8814 at 04:10 changed the cache key namespace from pcat:v7 to pcat:v8, invalidating all cached entries and causing cache hit rate to drop from 97% to 3%, which overwhelmed postgres-catalog with direct database queries.
remediation: Immediately revert the deploy dep-8814 to restore the old cache key namespace pcat:v7, which will allow the existing cache entries to be used and reduce database load to normal levels.
confident  : True (5 findings

## What you learned

- A reasoner can **choose its own fan-out** at runtime from a fixed vocabulary — the graph
  becomes a function of the input, not of the source file.
- **Cap it in code.** `MAX_LENSES = 6` is the difference between a plan and a shopping list;
  the model's list is also filtered against the vocabulary, so an invented lens cannot fire.
- Two incidents through the same entry point produced **different lens sets and different
  DAGs**, which is the thing chapter 04's hand-wired graph structurally cannot do.

**Next:** chapter 07 takes the notebook away. The same graph runs on a trigger with nobody
watching, and stops to ask a human before it does anything it cannot take back.